# Notebook 07: Hierarchical Contrastive Training

**Variant 4:** Multi-granularity loss that aligns queries to both API descriptions and category labels.

L = L_api(query, api_doc) + lambda * L_cat(query, category)

The category term acts as a regularizer, pushing queries toward the correct semantic region before fine-grained API matching.

**Prerequisite:** Run `01_index_apis.ipynb` first.

In [ ]:
import os, sys, json, random
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / '.git').exists())
PROJECT_DIR = REPO_ROOT / 'project'
sys.path.insert(0, str(PROJECT_DIR))

load_dotenv(REPO_ROOT / '.env')

TOOLBENCH_DIR = Path(os.environ.get('TOOLBENCH_DIR', str(REPO_ROOT / 'toolbench_data')))

from data.load_toolbench import load_api_corpus, load_eval_examples
from data.negative_mining import build_api_lookup, build_dfsdt_negatives
from models.embeddings import format_api_string

In [ ]:
corpus = load_api_corpus(TOOLBENCH_DIR / 'toolenv' / 'tools')
lookup = build_api_lookup(corpus)

TRAIN_PATH = TOOLBENCH_DIR / 'toolllama_G123_dfs_train.json'
assert TRAIN_PATH.exists(), f'Training file not found: {TRAIN_PATH}'

train_examples = load_eval_examples(TRAIN_PATH)
with open(TRAIN_PATH) as f:
    raw_train = json.load(f)

print(f'Corpus: {len(corpus)} APIs | Training: {len(train_examples)} examples')
print(f'Categories: {len(set(a["category"] for a in corpus))}')

## Hierarchical Loss

Combines API-level and category-level contrastive objectives. The category term uses in-batch negatives over category name embeddings.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class HierarchicalLoss(nn.Module):
    def __init__(self, model, scale=20.0, lambda_cat=0.1):
        super().__init__()
        self.model = model
        self.scale = scale
        self.lambda_cat = lambda_cat

    def forward(self, sentence_features, labels):
        reps = [self.model(sf)['sentence_embedding'] for sf in sentence_features]
        queries = reps[0]      # (B, D)
        categories = reps[2]   # (B, D)

        # API-level loss: query vs [positive_api, hard_neg1, ..., hard_negK]
        api_pool = torch.cat(reps[1:2] + reps[3:], dim=0)
        sim_api = F.cosine_similarity(queries.unsqueeze(1), api_pool.unsqueeze(0), dim=-1) * self.scale
        target = torch.arange(queries.size(0), device=queries.device)
        loss_api = F.cross_entropy(sim_api, target)

        # Category-level loss: query vs category names (in-batch negatives)
        sim_cat = F.cosine_similarity(queries.unsqueeze(1), categories.unsqueeze(0), dim=-1) * self.scale
        loss_cat = F.cross_entropy(sim_cat, target)

        return loss_api + self.lambda_cat * loss_cat

## Build Training Data

Each example: [query, positive_api, category_name, hard_neg1, ..., hard_neg7]

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample
from torch.utils.data import DataLoader

BASE_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
EPOCHS = 5
BATCH_SIZE = 64
N_HARD_NEGATIVES = 7
LAMBDA_CAT = 0.1

pairs = []
for ex in train_examples:
    raw_ex = raw_train[ex['raw_idx']]
    for name in ex['ground_truth_apis']:
        if name not in lookup:
            continue
        api = lookup[name]
        hard_negs = build_dfsdt_negatives(raw_ex, corpus, ex['ground_truth_apis'], lookup, n=N_HARD_NEGATIVES)
        neg_texts = [format_api_string(n) for n in hard_negs]
        if len(neg_texts) < N_HARD_NEGATIVES:
            continue
        pairs.append(InputExample(texts=[
            ex['user_query'],
            format_api_string(api),
            api['category'],
        ] + neg_texts))

random.shuffle(pairs)
print(f'Variant 4: {len(pairs)} training pairs')

## Train

In [ ]:
model = SentenceTransformer(BASE_MODEL)
loader = DataLoader(pairs, shuffle=True, batch_size=BATCH_SIZE)
loss = HierarchicalLoss(model, lambda_cat=LAMBDA_CAT)

model.fit(
    train_objectives=[(loader, loss)],
    epochs=EPOCHS,
    warmup_steps=int(0.1 * len(loader) * EPOCHS),
    output_path=str(PROJECT_DIR / 'checkpoints' / 'v4_hierarchical'),
    show_progress_bar=True,
)
print('Variant 4 saved to checkpoints/v4_hierarchical')